# NB13: CPTAC HRD label preparation

Prepares per-cohort HRD label CSVs for the CPTAC validation cohorts
(LUAD, LUSC, HNSCC, UCEC) by joining the published HRD scores from
Loeffler et al. (2024) BMC Biology (doi:10.1186/s12915-024-02022-9)
to the local CPTAC WSI manifest.

**Manuscript reference** (Methods + Supp Table S1):
- HRD threshold for CPTAC cohorts: scarHRD >= 42
- Cohorts and sizes:
  - CPTAC-LUAD : n=106
  - CPTAC-LUSC : n~106
  - CPTAC-HNSCC: n~89
  - CPTAC-UCEC : n=99 (excluded from primary AUROC reporting due to
                       insufficient HRD-positive events)

**Required env**:
- `WORKSPACE`
- `CPTAC_LABELS_XLSX` (Loeffler et al. 2024 HRD table; CPTAC sheet)
- `CPTAC_WSI_MANIFEST` (CSV/XLSX listing local CPTAC slides)

**Outputs** (per cohort): `labels/cptac_<cohort>_loeffler.csv`,
each with columns `patient`, `HRD_score`, `HRD_binary`.

In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
CPTAC_LABELS_XLSX = Path(os.environ["CPTAC_LABELS_XLSX"]).resolve()
CPTAC_WSI_MANIFEST = Path(os.environ["CPTAC_WSI_MANIFEST"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "cptac_labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked threshold for CPTAC (Methods)
HRD_THRESHOLD = 42

COHORTS = ["LUAD", "LUSC", "HNSCC", "UCEC"]

print(f"CPTAC_LABELS_XLSX  : {CPTAC_LABELS_XLSX}")
print(f"CPTAC_WSI_MANIFEST : {CPTAC_WSI_MANIFEST}")
print(f"threshold (CPTAC)  : scarHRD >= {HRD_THRESHOLD}")
print(f"cohorts            : {COHORTS}")


In [ ]:
def patient_token(s):
    s = str(s).strip().upper()
    s = re.sub(r"\.(SVS|TIF|TIFF|NDPI|MRXS|SCN|BIF)$", "", s)
    parts = s.split("-")
    if len(parts) >= 2 and re.fullmatch(r"[A-Z0-9]{3}", parts[0]) and re.fullmatch(r"\d{5}", parts[1]):
        return f"{parts[0]}-{parts[1]}"
    return s

xls = pd.ExcelFile(CPTAC_LABELS_XLSX)
sheet = "CPTAC" if "CPTAC" in xls.sheet_names else xls.sheet_names[0]
print(f"using sheet: {sheet} (available: {xls.sheet_names})")
loef = pd.read_excel(CPTAC_LABELS_XLSX, sheet_name=sheet)
print(f"raw rows: {len(loef)}  | columns: {list(loef.columns)[:12]}")


In [ ]:
def _norm(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower())

cmap = {_norm(c): c for c in loef.columns}
def pick(candidates):
    for k in candidates:
        if k in cmap:
            return cmap[k]
    return None

pat_col = pick(["patient", "patient_id", "case_id", "patient_identifier", "submitter_id"])
hrd_col = pick(["hrd_score", "hrd_sum", "hrd", "scarhrd", "scar_hrd"])
bin_col = pick(["hrd_binary", "hrd_bin", "hrd_status"])
cancer_col = pick(["cancer", "cancer_type", "tumor_type", "tumour_type", "cohort"])

assert pat_col is not None, f"no patient column found; columns are {list(loef.columns)}"
assert hrd_col is not None or bin_col is not None, "need either an HRD score or HRD binary column"
print(f"patient column : {pat_col}")
print(f"HRD score col  : {hrd_col}")
print(f"HRD binary col : {bin_col}")
print(f"cancer column  : {cancer_col}")


In [ ]:
loef["_patient"] = loef[pat_col].apply(patient_token)
if hrd_col is not None:
    loef["HRD_score"] = pd.to_numeric(loef[hrd_col], errors="coerce")
else:
    loef["HRD_score"] = np.nan

if bin_col is not None:
    loef["HRD_binary"] = pd.to_numeric(loef[bin_col], errors="coerce").fillna(-1).astype(int)
    loef.loc[loef["HRD_binary"] == -1, "HRD_binary"] = np.nan
else:
    loef["HRD_binary"] = (loef["HRD_score"] >= HRD_THRESHOLD).astype("Int64")

if cancer_col is not None:
    loef["_cancer"] = loef[cancer_col].astype(str).str.upper().str.strip()
else:
    loef["_cancer"] = "UNK"

print(f"normalized rows: {len(loef)}")
print(loef[["_patient", "_cancer", "HRD_score", "HRD_binary"]].head(8).to_string(index=False))


In [ ]:
if CPTAC_WSI_MANIFEST.suffix.lower() in (".xlsx", ".xls"):
    manifest = pd.read_excel(CPTAC_WSI_MANIFEST)
else:
    manifest = pd.read_csv(CPTAC_WSI_MANIFEST)

manifest.columns = [str(c) for c in manifest.columns]
mmap = {_norm(c): c for c in manifest.columns}
m_pat = mmap.get("patient") or mmap.get("patient_id") or mmap.get("case_id")
m_can = mmap.get("cancer") or mmap.get("cohort") or mmap.get("cancer_type") or mmap.get("tumor_type")
assert m_pat is not None and m_can is not None, (
    f"manifest must contain patient and cancer columns; got {list(manifest.columns)}"
)

manifest["_patient"] = manifest[m_pat].apply(patient_token)
manifest["_cancer"] = manifest[m_can].astype(str).str.upper().str.strip()
print(f"manifest rows: {len(manifest)}  | unique patients: {manifest['_patient'].nunique()}")
print(manifest["_cancer"].value_counts().head(8))


In [ ]:
summary = {}
for cohort in COHORTS:
    pat_pattern = r"HNSC" if cohort == "HNSCC" else cohort
    pts_in_manifest = sorted(set(
        manifest.loc[manifest["_cancer"].str.contains(pat_pattern, regex=True, case=False, na=False),
                     "_patient"]
    ))

    if cancer_col is not None:
        sub = loef[loef["_cancer"].str.contains(pat_pattern, regex=True, case=False, na=False)]
    else:
        sub = loef[loef["_patient"].isin(pts_in_manifest)]

    sub = sub.dropna(subset=["_patient"]).drop_duplicates("_patient")

    out_df = sub[["_patient", "HRD_score", "HRD_binary"]].rename(columns={"_patient": "patient"})
    out_path = LABELS_DIR / f"cptac_{cohort.lower()}_loeffler.csv"
    out_df.to_csv(out_path, index=False)

    in_manifest = out_df["patient"].isin(pts_in_manifest)
    n_match = int(in_manifest.sum())
    n_pos = int((out_df["HRD_binary"] == 1).sum())
    n_neg = int((out_df["HRD_binary"] == 0).sum())
    summary[cohort] = {
        "n_loeffler_rows": int(len(out_df)),
        "n_in_manifest": n_match,
        "n_HRD_pos_thr42": n_pos,
        "n_HRD_neg_thr42": n_neg,
        "out_path": str(out_path),
    }
    print(f"{cohort}: {len(out_df)} Loeffler rows, {n_match} in local manifest, "
          f"HRD+={n_pos}, HRD-={n_neg}  -> {out_path.name}")


In [ ]:
report = {
    "threshold_HRD": HRD_THRESHOLD,
    "source": "Loeffler CML et al. BMC Biology 22:225 (2024). doi:10.1186/s12915-024-02022-9",
    "cohorts": summary,
    "manuscript_refs": {
        "threshold": 42,
        "cohort_sizes_from_supp_S1": {"CPTAC_LUAD": 106, "CPTAC_UCEC": 99},
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
